<a href="https://colab.research.google.com/github/VayuSarangam/Default-Probabilities/blob/main/CRE_PD_Model_3_Versions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Commercial Real Estate Machine Learning PD Modeling

### Model Versions
- **Version 1: The 'Legacy Benchmark'** (Weighted RF)
- **Version 2: The 'Conservative Baseline'** (Unweighted RF)
- **Version 3: The 'Engineered Champion'** (Feature-Engineered RF)

### Input Factors (Features)
The models utilize the following loan-level and market factors to estimate risk:
- **Stressed LTV**: Loan-to-Value ratio under stress scenarios.
- **DSCR**: Debt Service Coverage Ratio.
- **Occupancy**: Current physical occupancy percentage.
- **Market Stress**: Regional economic risk indicator.
- **Property Risk Flag**: Indicator for high-risk property types.
- **Sponsor Risk Score**: Qualitative score of the borrower/sponsor strength.
- **Recovery Lag Months**: Expected time to recover capital post-default.
- **Cap Rate Expansion**: Potential shift in market capitalization rates.
- **Debt Yield**: Net Operating Income divided by the total loan amount.

### Model Outputs & Performance Metrics
The analysis generates the following results:
- **Probability of Default (PD)**: The estimated likelihood of a loan defaulting (0% to 100%).
- **Brier Score**: Measures the accuracy of probabilistic predictions (lower is better).
- **AUC / Gini**: Measures the model's ability to rank loans from low to high risk.
- **KS (Kolmogorov-Smirnov)**: Measures the maximum separation between default and non-default distributions.
- **Feature Importance**: Identifies which factors (e.g., LTV vs. Sponsor Risk) most influence the final PD.

In [87]:
# GENERATING THE SYNTHETIC LOAN DATASET
# =========================================================

# --- STEP 1: DEFINE SEED DATA ---
# These are 'base' examples used to build our larger dataset

stressed_ltv = np.array([0.72, 0.80, 0.64, 0.88, 0.76, 0.92, 0.68, 0.83, 0.79, 0.97])
dscr = np.array([1.45, 1.18, 1.72, 1.05, 1.33, 0.95, 1.58, 1.12, 1.25, 0.88])
occupancy = np.array([0.94, 0.88, 0.97, 0.82, 0.91, 0.78, 0.96, 0.85, 0.90, 0.74])
market_stress = np.array([0.20, 0.35, 0.12, 0.48, 0.28, 0.55, 0.15, 0.40, 0.32, 0.62])
property_risk_flag = np.array([0.35, 0.85, 0.00, 1.00, 0.25, 0.85, 0.35, 0.30, 0.00, 1.00])
sponsor_risk_score = np.array([3, 5, 2, 7, 4, 8, 2, 6, 4, 9], dtype=float)
recovery_lag_months = np.array([12, 18, 9, 24, 15, 30, 10, 21, 14, 36], dtype=float)
cap_rate_expansion = np.array([0.005, 0.010, 0.000, 0.018, 0.008, 0.025, 0.002, 0.014, 0.009, 0.030])
debt_yield = np.array([0.08, 0.08, 0.10, 0.08, 0.09, 0.07, 0.11, 0.08, 0.08, 0.06])

# Combine these into a single matrix (Seed Matrix)
X_seed = np.column_stack([
    stressed_ltv, dscr, occupancy, market_stress,
    property_risk_flag, sponsor_risk_score,
    recovery_lag_months, cap_rate_expansion, debt_yield
])

# --- STEP 2: DEFINE SCORING LOGIC (Baseline PD) ---
# We assign weight to each factor to calculate a 'Default Score' (Logit)

INTERCEPT = -4.75
WEIGHTS = {
    "LTV": 2.75, "DSCR": -1.05, "OCCUPANCY": -1.20, "MARKET": 1.10,
    "PROP_RISK": 0.85, "SPONSOR": 0.115, "REC_LAG": 0.0125,
    "CAP_EXP": 10.00, "DEBT_YIELD": -4.25
}

def calculate_baseline_pd(features):
    """Converts raw loan features into a Probability of Default (0 to 1)"""
    logit = (INTERCEPT
             + WEIGHTS["LTV"] * features[:, 0]
             + WEIGHTS["DSCR"] * features[:, 1]
             + WEIGHTS["OCCUPANCY"] * features[:, 2]
             + WEIGHTS["MARKET"] * features[:, 3]
             + WEIGHTS["PROP_RISK"] * features[:, 4]
             + WEIGHTS["SPONSOR"] * features[:, 5]
             + WEIGHTS["REC_LAG"] * features[:, 6]
             + WEIGHTS["CAP_EXP"] * features[:, 7]
             + WEIGHTS["DEBT_YIELD"] * features[:, 8])
    return 1 / (1 + np.exp(-logit))

# --- STEP 3: SCALE SEED DATA TO 3,000 LOANS ---
# We randomly sample from our 10 seeds and add 'noise' to make them unique

rng = np.random.default_rng(RANDOM_STATE)
indices = rng.integers(0, X_seed.shape[0], 3000)
X_synthetic = X_seed[indices].copy()

# Add slight randomness to each column so they aren't exact duplicates
noise_levels = [0.08, 0.18, 0.06, 0.12, 0.15, 1.25, 5.0, 0.006, 0.012]
for i in range(X_synthetic.shape[1]):
    X_synthetic[:, i] += rng.normal(0, noise_levels[i], 3000)

# Clip values to ensure they stay within realistic bounds
X_synthetic[:, 0] = np.clip(X_synthetic[:, 0], 0.40, 1.25) # LTV
X_synthetic[:, 1] = np.clip(X_synthetic[:, 1], 0.50, 2.50) # DSCR

# --- STEP 4: DETERMINE ACTUAL DEFAULTS (Target) ---
# We calculate the PD and then 'roll the dice' to see if the loan actually defaults

base_pd = calculate_baseline_pd(X_synthetic)
synthetic_logit = np.log(base_pd / (1 - base_pd))

# Apply 'Risk Penalties' for bad combinations (e.g., High LTV + Low DSCR)
high_risk_trigger = ((X_synthetic[:, 0] > 0.85) & (X_synthetic[:, 1] < 1.10)).astype(int)
synthetic_logit += (1.10 * high_risk_trigger)

final_pd = 1 / (1 + np.exp(-synthetic_logit))
y_synthetic = rng.binomial(1, final_pd)

# --- STEP 5: EXPORT TO EXCEL ---

df_export = pd.DataFrame(X_synthetic, columns=BASE_FEATURES)
df_export[TARGET_COLUMN] = y_synthetic
df_export.insert(0, "Loan ID", [f"SYNTH-{i+1:04d}" for i in range(len(df_export))])

output_path = "/content/Name of new loan list.xlsx"
df_export.to_excel(output_path, index=False, sheet_name="ML_Test_Data")

# --- STEP 6: SPLIT FOR TRAINING ---

X_dev_train, X_dev_internal, y_dev_train, y_dev_internal = train_test_split(
    X_synthetic, y_synthetic, test_size=0.25, random_state=RANDOM_STATE, stratify=y_synthetic
)

print(f"Success! 3,000 synthetic loans saved to: {output_path}")
print(f"Default Rate in sample: {np.mean(y_synthetic):.2%}")

Success! 3,000 synthetic loans saved to: /content/Name of new loan list.xlsx
Default Rate in sample: 12.60%


In [88]:
# =========================================================
# CELL 1 — SETUP AND CONFIGURATION
# =========================================================

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    brier_score_loss,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)

RANDOM_STATE = 42
# Path to the uploaded file
FILE_PATH = Path("/content/Name of new loan list.xlsx")
# Updated to match the actual sheet name found in your file
SHEET_NAME = "ML_Test_Data"
TARGET_COLUMN = "Actual Default"

BASE_FEATURES = [
    "Stressed LTV",
    "DSCR",
    "Occupancy",
    "Market Stress",
    "Property Risk Flag",
    "Sponsor Risk Score",
    "Recovery Lag Months",
    "Cap Rate Expansion",
    "Debt Yield",
]

print("Setup complete.")
print(f"Targeting File: {FILE_PATH}")
print(f"Targeting Sheet: {SHEET_NAME}")
print("Excel file exists:", FILE_PATH.exists())

Setup complete.
Targeting File: /content/Name of new loan list.xlsx
Targeting Sheet: ML_Test_Data
Excel file exists: True


In [89]:
# =========================================================
# CELL 3 — LOAD EXTERNAL VALIDATION / CALIBRATION DATA
# =========================================================
import openpyxl
if not FILE_PATH.exists():
    raise FileNotFoundError(
        f"Excel file not found: {FILE_PATH}. Update FILE_PATH in Cell 1."
    )

validation_loans = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)
validation_loans.columns = validation_loans.columns.astype(str).str.strip()

required_columns = ["Loan ID", TARGET_COLUMN] + BASE_FEATURES
missing_columns = [c for c in required_columns if c not in validation_loans.columns]

if missing_columns:
    raise ValueError(f"Missing Excel columns: {missing_columns}")

X_validation = validation_loans[BASE_FEATURES].astype(float).to_numpy()
y_validation = validation_loans[TARGET_COLUMN].astype(int).to_numpy()

print("External validation/calibration data loaded.")
print("Loans:", len(validation_loans))
print(f"Observed default rate: {np.mean(y_validation):.2%}")
print(validation_loans.head())

External validation/calibration data loaded.
Loans: 3000
Observed default rate: 12.60%
      Loan ID  Stressed LTV      DSCR  Occupancy  Market Stress  \
0  SYNTH-0001      0.797037  1.518704   0.961055       0.255078   
1  SYNTH-0002      0.772031  1.110393   0.840745       0.339223   
2  SYNTH-0003      0.611699  1.775539   1.022956       0.274839   
3  SYNTH-0004      0.796776  1.218944   0.862051       0.335880   
4  SYNTH-0005      0.839084  1.507511   0.917210       0.592404   

   Property Risk Flag  Sponsor Risk Score  Recovery Lag Months  \
0            0.570651            4.934878             7.207182   
1            0.463520            3.996617            25.119963   
2            0.587908            1.771022             5.036687   
3            0.285545            2.542113            15.270077   
4            0.244640            3.060454            18.280641   

   Cap Rate Expansion  Debt Yield  Actual Default  
0            0.004803    0.085731               0  
1        

In [90]:
# =========================================================
# CELL 3 — LOAD EXTERNAL VALIDATION / CALIBRATION DATA
# =========================================================
import openpyxl
if not FILE_PATH.exists():
    raise FileNotFoundError(
        f"Excel file not found: {FILE_PATH}. Update FILE_PATH in Cell 1."
    )

validation_loans = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)
validation_loans.columns = validation_loans.columns.astype(str).str.strip()

required_columns = ["Loan ID", TARGET_COLUMN] + BASE_FEATURES
missing_columns = [c for c in required_columns if c not in validation_loans.columns]

if missing_columns:
    raise ValueError(f"Missing Excel columns: {missing_columns}")

X_validation = validation_loans[BASE_FEATURES].astype(float).to_numpy()
y_validation = validation_loans[TARGET_COLUMN].astype(int).to_numpy()

print("External validation/calibration data loaded.")
print("Loans:", len(validation_loans))
print(f"Observed default rate: {np.mean(y_validation):.2%}")
print(validation_loans.head())

External validation/calibration data loaded.
Loans: 3000
Observed default rate: 12.60%
      Loan ID  Stressed LTV      DSCR  Occupancy  Market Stress  \
0  SYNTH-0001      0.797037  1.518704   0.961055       0.255078   
1  SYNTH-0002      0.772031  1.110393   0.840745       0.339223   
2  SYNTH-0003      0.611699  1.775539   1.022956       0.274839   
3  SYNTH-0004      0.796776  1.218944   0.862051       0.335880   
4  SYNTH-0005      0.839084  1.507511   0.917210       0.592404   

   Property Risk Flag  Sponsor Risk Score  Recovery Lag Months  \
0            0.570651            4.934878             7.207182   
1            0.463520            3.996617            25.119963   
2            0.587908            1.771022             5.036687   
3            0.285545            2.542113            15.270077   
4            0.244640            3.060454            18.280641   

   Cap Rate Expansion  Debt Yield  Actual Default  
0            0.004803    0.085731               0  
1        

In [91]:
import pandas as pd

# Assuming FILE_PATH is correctly defined in Cell 1
if FILE_PATH.exists():
    xls = pd.ExcelFile(FILE_PATH)
    print(f"Sheets in '{FILE_PATH.name}': {xls.sheet_names}")
else:
    print(f"Error: Excel file not found at {FILE_PATH}. Please ensure FILE_PATH in Cell 1 is correct.")

Sheets in 'Name of new loan list.xlsx': ['ML_Test_Data']


In [92]:
# =========================================================
# CELL 3 — LOAD EXTERNAL VALIDATION / CALIBRATION DATA
# =========================================================
import openpyxl
if not FILE_PATH.exists():
    raise FileNotFoundError(
        f"Excel file not found: {FILE_PATH}. Update FILE_PATH in Cell 1."
    )

validation_loans = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)
validation_loans.columns = validation_loans.columns.astype(str).str.strip()

required_columns = ["Loan ID", TARGET_COLUMN] + BASE_FEATURES
missing_columns = [c for c in required_columns if c not in validation_loans.columns]

if missing_columns:
    raise ValueError(f"Missing Excel columns: {missing_columns}")

X_validation = validation_loans[BASE_FEATURES].astype(float).to_numpy()
y_validation = validation_loans[TARGET_COLUMN].astype(int).to_numpy()

print("External validation/calibration data loaded.")
print("Loans:", len(validation_loans))
print(f"Observed default rate: {np.mean(y_validation):.2%}")
print(validation_loans.head())

External validation/calibration data loaded.
Loans: 3000
Observed default rate: 12.60%
      Loan ID  Stressed LTV      DSCR  Occupancy  Market Stress  \
0  SYNTH-0001      0.797037  1.518704   0.961055       0.255078   
1  SYNTH-0002      0.772031  1.110393   0.840745       0.339223   
2  SYNTH-0003      0.611699  1.775539   1.022956       0.274839   
3  SYNTH-0004      0.796776  1.218944   0.862051       0.335880   
4  SYNTH-0005      0.839084  1.507511   0.917210       0.592404   

   Property Risk Flag  Sponsor Risk Score  Recovery Lag Months  \
0            0.570651            4.934878             7.207182   
1            0.463520            3.996617            25.119963   
2            0.587908            1.771022             5.036687   
3            0.285545            2.542113            15.270077   
4            0.244640            3.060454            18.280641   

   Cap Rate Expansion  Debt Yield  Actual Default  
0            0.004803    0.085731               0  
1        

In [95]:
# =========================================================
# CELL 4 — METRICS & CALIBRATION UTILITIES
# =========================================================

def probability_to_logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p)).reshape(-1, 1)

def calculate_pd_metrics(actuals, probs):
    actuals, probs = np.asarray(actuals, dtype=int), np.asarray(probs, dtype=float)
    auc = roc_auc_score(actuals, probs)
    fpr, tpr, _ = roc_curve(actuals, probs)
    dr = np.mean(actuals)
    brier = brier_score_loss(actuals, probs)
    return {
        "AUC": auc, "Gini": 2 * auc - 1, "KS": np.max(tpr - fpr),
        "Brier Score": brier, "Brier Skill Score": 1 - (brier / (dr * (1 - dr))),
        "Average Predicted PD": np.mean(probs)
    }

def crossfit_platt_calibration(raw_pd, actuals, n_splits=5):
    # Calculates out-of-fold calibrated PDs and a final model
    raw_pd, actuals = np.asarray(raw_pd), np.asarray(actuals)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof_pd = np.zeros(len(actuals))
    for tr, ts in cv.split(raw_pd, actuals):
        cal = LogisticRegression(C=1.0, solver="lbfgs", max_iter=2000, random_state=RANDOM_STATE)
        cal.fit(probability_to_logit(raw_pd[tr]), actuals[tr])
        oof_pd[ts] = cal.predict_proba(probability_to_logit(raw_pd[ts]))[:, 1]
    final_cal = LogisticRegression(C=1.0, solver="lbfgs", max_iter=2000, random_state=RANDOM_STATE)
    final_cal.fit(probability_to_logit(raw_pd), actuals)
    return oof_pd, final_cal

def calculate_threshold_metrics(actuals, probs, t):
    # Stats for a specific decision threshold (t)
    flags = (np.asarray(probs) >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(actuals, flags, labels=[0, 1]).ravel()
    rec = tp / (tp + fn) if (tp + fn) else 0
    spec = tn / (tn + fp) if (tn + fp) else 0
    return {
        "Threshold": t, "True Negatives": tn, "False Positives": fp,
        "False Negatives": fn, "True Positives": tp, "Loans Flagged": tp + fp,
        "Precision": precision_score(actuals, flags, zero_division=0),
        "Recall": rec, "Specificity": spec, "Accuracy": accuracy_score(actuals, flags),
        "Youden J": rec + spec - 1
    }

def select_watchlist_threshold(actuals, probs, target_recall=0.75):
    cands = np.unique(np.concatenate([np.arange(0.01, 0.61, 0.005), probs]))
    table = pd.DataFrame([calculate_threshold_metrics(actuals, probs, t) for t in cands])
    eligible = table[table["Recall"] >= target_recall]
    res = eligible.sort_values(["Specificity", "Precision"], ascending=False).iloc[0] if not eligible.empty else table.loc[table["Recall"].idxmax()]
    return res, table

def format_metric_table(df):
    fmts = {"AUC": "{:.4f}", "Gini": "{:.4f}", "KS": "{:.4f}", "Brier Score": "{:.4f}",
            "Average Predicted PD": "{:.2%}", "Threshold": "{:.2%}", "Recall": "{:.2%}",
            "Specificity": "{:.2%}", "Accuracy": "{:.2%}", "Precision": "{:.2%}"}
    return df.to_string(index=False, formatters={k: v.format for k, v in fmts.items() if k in df.columns})

print("Utility functions ready.")

Utility functions ready.


In [102]:
# =========================================================
# CELL 5 — VERSION 1: THE 'LEGACY BENCHMARK'
# =========================================================
# 1. Initialize the Legacy Benchmark model
model_legacy_benchmark = RandomForestClassifier(
    n_estimators=400, max_depth=6, min_samples_leaf=15,
    class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
)

# 2. Train using dev sample
model_legacy_benchmark.fit(X_dev_train, y_dev_train)

# 3. Generate PDs
v1_internal_pd = model_legacy_benchmark.predict_proba(X_dev_internal)[:, 1]
v1_validation_raw_pd = model_legacy_benchmark.predict_proba(X_validation)[:, 1]

# 4. Calibration
v1_validation_cal_pd, calib_legacy_benchmark = crossfit_platt_calibration(v1_validation_raw_pd, y_validation)

# 5. Store in validation dataframe
validation_loans["Legacy Benchmark PD"] = v1_validation_raw_pd
validation_loans["Legacy Benchmark Calibrated PD"] = v1_validation_cal_pd

print("VERSION 1: LEGACY BENCHMARK INTERNAL TEST")
print(format_metric_table(pd.DataFrame([{
    "Model": "Legacy Benchmark",
    **calculate_pd_metrics(y_dev_internal, v1_internal_pd),
}])))

VERSION 1: LEGACY BENCHMARK INTERNAL TEST
           Model    AUC   Gini     KS Brier Score  Brier Skill Score Average Predicted PD
Legacy Benchmark 0.8972 0.7945 0.6971      0.1104           -0.00729               26.82%


In [103]:
# =========================================================
# CELL 6 — VERSION 2: THE 'CONSERVATIVE BASELINE'
# =========================================================
# 1. Initialize the Conservative Baseline (Unweighted)
base_rf_v2 = RandomForestClassifier(
    n_estimators=600, max_depth=5, min_samples_leaf=20,
    max_features="sqrt", class_weight=None, random_state=RANDOM_STATE, n_jobs=-1
)

# 2. Wrap with CalibratedClassifierCV
model_conservative_baseline = CalibratedClassifierCV(estimator=base_rf_v2, method="sigmoid", cv=5)
model_conservative_baseline.fit(X_dev_train, y_dev_train)

# 3. Predict and calibrate
v2_validation_raw_pd = model_conservative_baseline.predict_proba(X_validation)[:, 1]
v2_validation_cal_pd, calib_conservative_baseline = crossfit_platt_calibration(v2_validation_raw_pd, y_validation)

# 4. Store results
validation_loans["Conservative Baseline PD"] = v2_validation_raw_pd
validation_loans["Conservative Baseline Calibrated PD"] = v2_validation_cal_pd

print("VERSION 2 — CONSERVATIVE BASELINE VALIDATION")
print(format_metric_table(pd.DataFrame([
    {"Model": "Conservative Baseline (Raw)", **calculate_pd_metrics(y_validation, v2_validation_raw_pd)},
    {"Model": "Conservative Baseline (Platt)", **calculate_pd_metrics(y_validation, v2_validation_cal_pd)}
])))

VERSION 2 — CONSERVATIVE BASELINE VALIDATION
                        Model    AUC   Gini     KS Brier Score  Brier Skill Score Average Predicted PD
  Conservative Baseline (Raw) 0.9163 0.8326 0.6942      0.0715           0.351090               12.70%
Conservative Baseline (Platt) 0.9131 0.8263 0.6953      0.0718           0.347558               12.61%


In [104]:
# =========================================================
# CELL 7 — VERSION 3: THE 'ENGINEERED CHAMPION'
# =========================================================
def create_engineered_feature_matrix(X_input):
    X = np.asarray(X_input, dtype=float)
    ltv, dscr, occ, mkt, spon, cap, dy = X[:,0], X[:,1], X[:,2], X[:,3], X[:,5], X[:,7], X[:,8]
    safe_dscr = np.maximum(dscr, 0.25)
    eng = np.column_stack([
        ltv / safe_dscr, (1 - occ) * mkt, ltv * cap, np.maximum(0, 0.08 - dy),
        np.maximum(0, 1.20 - dscr), np.maximum(0, ltv - 0.80), np.maximum(0, 0.85 - occ),
        ((ltv >= 0.85) & (dscr <= 1.10)).astype(int), ((occ <= 0.80) & (mkt >= 0.50)).astype(int),
        (spon / 10) * mkt
    ])
    return np.column_stack([X, eng])

X_dev_train_v3 = create_engineered_feature_matrix(X_dev_train)
X_validation_v3 = create_engineered_feature_matrix(X_validation)

# 1. Search for best params (Lite mode)
search_v3 = RandomizedSearchCV(
    estimator=RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    param_distributions={"max_depth": [5, 7], "min_samples_leaf": [15, 25], "max_features": ["sqrt", 0.50]},
    n_iter=2, scoring="roc_auc", cv=3, n_jobs=-1, random_state=RANDOM_STATE
)
search_v3.fit(X_dev_train_v3, y_dev_train)

# 2. Train the Engineered Champion
model_engineered_champion = CalibratedClassifierCV(
    estimator=RandomForestClassifier(n_estimators=500, **search_v3.best_params_, n_jobs=-1), cv=3
)
model_engineered_champion.fit(X_dev_train_v3, y_dev_train)

# 3. Predict and calibrate
v3_validation_raw_pd = model_engineered_champion.predict_proba(X_validation_v3)[:, 1]
v3_validation_cal_pd, calib_engineered_champion = crossfit_platt_calibration(v3_validation_raw_pd, y_validation)

validation_loans["Engineered Champion PD"] = v3_validation_raw_pd
validation_loans["Engineered Champion Calibrated PD"] = v3_validation_cal_pd

print("VERSION 3 — ENGINEERED CHAMPION VALIDATION")
print(format_metric_table(pd.DataFrame([
    {"Model": "Engineered Champion (Raw)", **calculate_pd_metrics(y_validation, v3_validation_raw_pd)},
    {"Model": "Engineered Champion (Platt)", **calculate_pd_metrics(y_validation, v3_validation_cal_pd)}
])))

VERSION 3 — ENGINEERED CHAMPION VALIDATION
                      Model    AUC   Gini     KS Brier Score  Brier Skill Score Average Predicted PD
  Engineered Champion (Raw) 0.9118 0.8235 0.6879      0.0729           0.337830               12.61%
Engineered Champion (Platt) 0.9083 0.8167 0.6879      0.0733           0.333976               12.61%


In [105]:
# =========================================================
# CELL 8 — CLEAN THREE-VERSION COMPARISON
# =========================================================
version_predictions = {
    "Legacy Benchmark": validation_loans["Legacy Benchmark Calibrated PD"].to_numpy(),
    "Conservative Baseline": validation_loans["Conservative Baseline Calibrated PD"].to_numpy(),
    "Engineered Champion": validation_loans["Engineered Champion Calibrated PD"].to_numpy(),
}

comparison_rows = [{"Model": name, **calculate_pd_metrics(y_validation, probs)} for name, probs in version_predictions.items()]
comparison_df = pd.DataFrame(comparison_rows).sort_values(by="Brier Score")

print("THREE-VERSION PD MODEL COMPARISON")
print("=" * 100)
print(format_metric_table(comparison_df))

THREE-VERSION PD MODEL COMPARISON
                Model    AUC   Gini     KS Brier Score  Brier Skill Score Average Predicted PD
     Legacy Benchmark 0.9294 0.8588 0.7361      0.0669           0.392321               12.61%
Conservative Baseline 0.9131 0.8263 0.6953      0.0718           0.347558               12.61%
  Engineered Champion 0.9083 0.8167 0.6879      0.0733           0.333976               12.61%


In [106]:
# =========================================================
# CELL 9 — CHAMPION INTERPRETATION
# =========================================================
CHAMPION_COL = "Engineered Champion Calibrated PD"

# Decile Analysis
calib_view = validation_loans.copy()
calib_view["Decile"] = pd.qcut(calib_view[CHAMPION_COL], q=10, duplicates="drop")
decile_table = calib_view.groupby("Decile", observed=False).agg(
    Count=("Loan ID", "count"),
    Avg_Pred_PD=(CHAMPION_COL, "mean"),
    Obs_Def_Rate=(TARGET_COLUMN, "mean")
).reset_index()

print(f"CALIBRATION DECILES: {CHAMPION_COL}")
print(decile_table.to_string(index=False))

# Feature Importance for Champion
imp_model = RandomForestClassifier(n_estimators=500, **search_v3.best_params_, n_jobs=-1)
imp_model.fit(X_dev_train_v3, y_dev_train)

importance_df = pd.DataFrame({"Feature": ALL_V3_FEATURES, "Importance": imp_model.feature_importances_}).sort_values("Importance", ascending=False)
print("\nENGINEERED CHAMPION FEATURE IMPORTANCE")
print(importance_df.head(10).to_string(index=False))

CALIBRATION DECILES: Engineered Champion Calibrated PD
          Decile  Count  Avg_Pred_PD  Obs_Def_Rate
 (0.0197, 0.022]    300     0.021244      0.000000
 (0.022, 0.0235]    300     0.022780      0.003333
(0.0235, 0.0251]    300     0.024321      0.006667
(0.0251, 0.0265]    300     0.025668      0.013333
(0.0265, 0.0297]    300     0.027849      0.010000
(0.0297, 0.0378]    300     0.033264      0.043333
 (0.0378, 0.064]    300     0.047730      0.056667
  (0.064, 0.164]    300     0.107009      0.173333
  (0.164, 0.433]    300     0.284941      0.330000
  (0.433, 0.899]    300     0.666566      0.623333

ENGINEERED CHAMPION FEATURE IMPORTANCE
                     Feature  Importance
       LTV / DSCR Risk Ratio    0.161606
   High LTV + Weak DSCR Flag    0.128974
    LTV × Cap Rate Expansion    0.100372
                Stressed LTV    0.087193
         Recovery Lag Months    0.076139
          Sponsor Risk Score    0.074620
                  LTV Excess    0.070642
Sponsor Risk × M